# Activation-space separability — do forget & retain form separate clusters?

CASAL-style PCA of the residual-stream activations (last question token), forget (wmdp-bio, red) vs retain (mmlu college bio, blue). If the two colors separate, there's a direction that distinguishes them despite shared experts.

Needs `run_results/separability/Qwen3-30B-A3B/activations.npz` + `silhouette.json` (scp/git-pull from the cluster after the run). With only 4 smoke-test prompts the clusters are sparse — use the full 160-sample run for real plots.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

R = Path('.').resolve().parent / 'run_results/separability/Qwen3-30B-A3B'
d = np.load(R / 'activations.npz')
Af = d['forget'].astype(np.float32)    # [L, n_forget, hidden]
Ar = d['retain'].astype(np.float32)    # [L, n_retain, hidden]
sil = json.load(open(R / 'silhouette.json'))
L, nf, nr = Af.shape[0], Af.shape[1], Ar.shape[1]
print(f'layers={L}  forget={nf}  retain={nr}  hidden={Af.shape[2]}')

def pca2(X):
    Xc = X - X.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

## Separability per layer (silhouette)

In [ ]:
sf = np.array(sil['silhouette_full']); s2 = np.array(sil['silhouette_pca2'])
plt.figure(figsize=(11,4))
plt.plot(sf, marker='.', label='full-dim (biased low in high-d)')
plt.plot(s2, marker='.', label='PCA-2D')
plt.axhline(0, c='grey', lw=.8)
plt.xlabel('layer'); plt.ylabel('silhouette (forget vs retain)')
plt.title('Separability per layer  (higher = more separable, ~0 = entangled)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
best = int(s2.argmax())
print(f'most separable layer (PCA-2D): {best}  silhouette={s2[best]:.3f}')

## Look at the clusters — one layer

In [ ]:
def show_clusters(l):
    Z = pca2(np.concatenate([Af[l], Ar[l]], 0))
    plt.figure(figsize=(6,5))
    plt.scatter(Z[:nf,0], Z[:nf,1], s=20, c='#cc4444', alpha=.7, label='forget (wmdp-bio)')
    plt.scatter(Z[nf:,0], Z[nf:,1], s=20, c='#4488aa', alpha=.7, label='retain (mmlu bio)')
    plt.title(f'Layer {l}   (silhouette {sil["silhouette_pca2"][l]:.2f})')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.tight_layout(); plt.show()

show_clusters(best)          # the most separable layer

## Grid across layers

In [ ]:
layers = [0, 8, 16, 24, 32, 40, 44, L-1]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, l in zip(axes.ravel(), layers):
    Z = pca2(np.concatenate([Af[l], Ar[l]], 0))
    ax.scatter(Z[:nf,0], Z[:nf,1], s=10, c='#cc4444', alpha=.7)
    ax.scatter(Z[nf:,0], Z[nf:,1], s=10, c='#4488aa', alpha=.7)
    ax.set_title(f'L{l}  sil={sil["silhouette_pca2"][l]:.2f}')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('forget (red) vs retain (blue) — PCA of residual activations per layer')
plt.tight_layout(); plt.show()